# Industrial Energy Forecasting & Early-Warning System

**Objective:** Predict the next 15-minute industrial energy demand, quantify forecast error, explain the drivers, and publish a Tableau-ready operational output.

**Flow:** Data Quality → Feature Logic → Demand Analysis → Time-Aware Validation → Model Selection → Final Holdout → Explainability → Early Warning → Tableau Output → Business Action


In [ ]:
import io
import os
import zipfile
import urllib.request
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

PLOT_TEMPLATE = "plotly_white"
PALETTE = {
    "navy": "#183B56",
    "blue": "#2F6BFF",
    "teal": "#16A085",
    "orange": "#F39C12",
    "red": "#D64545",
    "green": "#2E8B57",
    "gray": "#6B7280",
    "light": "#F5F7FA"
}

def style_fig(fig, title, height=480):
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=20)),
        template=PLOT_TEMPLATE,
        height=height,
        margin=dict(l=55, r=35, t=70, b=55),
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1),
        hovermode="x unified"
    )
    return fig

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

## 1. Data Acquisition & Validation

In [ ]:
EXPECTED_COLUMNS = [
    "date",
    "Usage_kWh",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "CO2(tCO2)",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM",
    "WeekStatus",
    "Day_of_week",
    "Load_Type"
]

UCI_URL = "https://archive.ics.uci.edu/static/public/851/steel+industry+energy+consumption.zip"
MIRROR_URL = "https://huggingface.co/spaces/praveenreddymudupu/Steel_Industry_Regression/resolve/main/Steel_industry_data.csv"

def normalize_schema(data):
    data = data.copy()
    data.columns = [str(c).strip() for c in data.columns]
    missing = [c for c in EXPECTED_COLUMNS if c not in data.columns]
    if missing:
        raise ValueError(f"Schema mismatch. Missing columns: {missing}")
    return data[EXPECTED_COLUMNS]

def load_dataset():
    local_paths = [
        "/content/Steel_industry_data.csv",
        "/mnt/data/Steel_industry_data.csv"
    ]

    for path in local_paths:
        if os.path.exists(path):
            return normalize_schema(pd.read_csv(path)), "local"

    try:
        raw = urllib.request.urlopen(UCI_URL, timeout=30).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            csv_files = [n for n in z.namelist() if n.lower().endswith(".csv")]
            if not csv_files:
                raise ValueError("No CSV found in UCI archive.")
            with z.open(csv_files[0]) as f:
                return normalize_schema(pd.read_csv(f)), "UCI"
    except Exception:
        pass

    try:
        with urllib.request.urlopen(MIRROR_URL, timeout=30) as response:
            return normalize_schema(pd.read_csv(response)), "mirror"
    except Exception as exc:
        raise RuntimeError(
            "Dataset acquisition failed. Restore internet access or place "
            "'Steel_industry_data.csv' in the runtime."
        ) from exc

df, data_source = load_dataset()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = (
    df.dropna(subset=["date", "Usage_kWh"])
      .sort_values("date")
      .drop_duplicates(subset="date", keep="first")
      .reset_index(drop=True)
)

print(f"Source: {data_source}")
print(f"Rows: {len(df):,}")
print(f"Period: {df['date'].min()} → {df['date'].max()}")

In [ ]:
quality = pd.DataFrame({
    "Metric": [
        "Rows", "Columns", "Duplicate timestamps", "Missing values",
        "Unique load types", "Unique weekdays"
    ],
    "Value": [
        len(df),
        df.shape[1],
        int(df["date"].duplicated().sum()),
        int(df.isna().sum().sum()),
        int(df["Load_Type"].nunique()),
        int(df["Day_of_week"].nunique())
    ]
})
display(quality)

## 2. Feature Engineering — Only Features With a Forecasting Reason

The target is the **next 15-minute Usage_kWh**. Every feature below is available at the forecast timestamp; no future value is used.

| Feature group | Why it is used |
|---|---|
| Current usage | The most recent load level is usually the strongest short-horizon signal. |
| Usage lags | Capture short-term persistence and repeating operating cycles. |
| Rolling statistics | Capture recent level, volatility and local operating regime. |
| Reactive power / power factor | Describe electrical operating conditions linked to plant load. |
| CO₂ | Acts as an operational intensity signal associated with energy use. |
| Time + cyclic time | Capture daily and weekly demand patterns without treating 23:45 and 00:00 as far apart. |
| Load / day status | Capture production and operating-state differences. |


In [ ]:
df["target_next_15m_kWh"] = df["Usage_kWh"].shift(-1)

df["hour"] = df["date"].dt.hour
df["minute"] = df["date"].dt.minute
df["day_of_week_num"] = df["date"].dt.dayofweek
minutes = df["hour"] * 60 + df["minute"]
df["time_sin"] = np.sin(2 * np.pi * minutes / 1440)
df["time_cos"] = np.cos(2 * np.pi * minutes / 1440)

df["usage_current"] = df["Usage_kWh"]
df["usage_lag_1"] = df["Usage_kWh"].shift(1)
df["usage_lag_4"] = df["Usage_kWh"].shift(4)
df["usage_lag_12"] = df["Usage_kWh"].shift(12)
df["usage_lag_96"] = df["Usage_kWh"].shift(96)
df["usage_change_1"] = df["Usage_kWh"].diff(1)
df["usage_change_4"] = df["Usage_kWh"].diff(4)
df["usage_rolling_mean_4"] = df["Usage_kWh"].rolling(4).mean()
df["usage_rolling_mean_12"] = df["Usage_kWh"].rolling(12).mean()
df["usage_rolling_mean_96"] = df["Usage_kWh"].rolling(96).mean()
df["usage_rolling_std_12"] = df["Usage_kWh"].rolling(12).std()
df["usage_rolling_std_96"] = df["Usage_kWh"].rolling(96).std()

df = df.dropna().reset_index(drop=True)
print(f"Modeling rows: {len(df):,}")


### 2.1 Feature Sanity Check

The model should use signals that are known **before** the next 15-minute interval. The table below makes that contract visible before training.


In [ ]:
feature_contract = pd.DataFrame({
    "Feature": [
        "usage_current", "usage_lag_1", "usage_lag_4", "usage_lag_12", "usage_lag_96",
        "usage_change_1", "usage_change_4", "usage_rolling_mean_4", "usage_rolling_mean_12",
        "usage_rolling_mean_96", "usage_rolling_std_12", "usage_rolling_std_96"
    ],
    "Forecasting reason": [
        "Latest observed demand level", "Immediate persistence", "One-hour operating cycle",
        "Three-hour operating cycle", "24-hour operating cycle", "Very recent load direction",
        "Short-term load direction", "Recent level", "Recent short-term level",
        "Daily operating level", "Short-term volatility", "Daily volatility regime"
    ]
})
display(feature_contract)


## 3. Demand Profile Analysis

In [ ]:
# Full-series trend with a 24-hour rolling mean
trend = df[["date", "Usage_kWh"]].copy()
trend["rolling_24h"] = trend["Usage_kWh"].rolling(96).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trend["date"], y=trend["Usage_kWh"],
    mode="lines", name="15-min Usage",
    line=dict(width=0.8, color=PALETTE["blue"])
))
fig.add_trace(go.Scatter(
    x=trend["date"], y=trend["rolling_24h"],
    mode="lines", name="24-hour Rolling Mean",
    line=dict(width=2.5, color=PALETTE["navy"])
))
style_fig(fig, "Industrial Energy Demand — Full Operating History", 520)
fig.update_xaxes(rangeslider_visible=True)
fig.update_yaxes(title="Energy Consumption (kWh)")
fig.show()

In [ ]:
hour_day = (
    df.assign(Hour=df["hour"])
      .pivot_table(index="Day_of_week", columns="Hour", values="Usage_kWh", aggfunc="mean")
      .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
)

fig = px.imshow(
    hour_day,
    aspect="auto",
    color_continuous_scale="Blues",
    labels=dict(x="Hour of Day", y="Day", color="Mean kWh"),
)
style_fig(fig, "Operating Demand Heatmap — Day × Hour", 500)
fig.show()

In [ ]:
load_profile = (
    df.groupby("Load_Type", as_index=False)
      .agg(
          Mean_kWh=("Usage_kWh","mean"),
          Median_kWh=("Usage_kWh","median"),
          P95_kWh=("Usage_kWh", lambda x: x.quantile(0.95))
      )
      .sort_values("Mean_kWh", ascending=False)
)

display(load_profile.round(2))

fig = go.Figure()
for col, color in [
    ("Mean_kWh", PALETTE["blue"]),
    ("Median_kWh", PALETTE["teal"]),
    ("P95_kWh", PALETTE["orange"])
]:
    fig.add_trace(go.Bar(
        x=load_profile["Load_Type"], y=load_profile[col],
        name=col.replace("_kWh","").replace("_"," "),
        marker_color=color
    ))
fig.update_layout(barmode="group")
style_fig(fig, "Demand Intensity by Load Type", 460)
fig.update_yaxes(title="Energy Consumption (kWh)")
fig.show()

In [ ]:
sample = df.sample(min(10000, len(df)), random_state=42)

fig = px.scatter(
    sample,
    x="CO2(tCO2)",
    y="Usage_kWh",
    color="Load_Type",
    opacity=0.55,
    color_discrete_sequence=[PALETTE["blue"], PALETTE["teal"], PALETTE["orange"]]
)
style_fig(fig, "Energy Consumption vs CO₂ Emissions", 500)
fig.update_xaxes(title="CO₂ (tCO₂)")
fig.update_yaxes(title="Energy Consumption (kWh)")
fig.show()

In [ ]:
corr_cols = [
    "Usage_kWh",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "CO2(tCO2)",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM",
    "usage_lag_1",
    "usage_rolling_mean_12"
]
corr = df[corr_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)
style_fig(fig, "Correlation Structure — Energy and Operational Signals", 560)
fig.show()

## 4. Forecast Dataset & Chronological Evaluation

In [ ]:
FEATURES = [
    "Usage_kWh",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "CO2(tCO2)",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM",
    "WeekStatus",
    "Day_of_week",
    "Load_Type",
    "usage_current",
    "usage_lag_1",
    "usage_lag_4",
    "usage_lag_12",
    "usage_lag_96",
    "usage_change_1",
    "usage_change_4",
    "usage_rolling_mean_4",
    "usage_rolling_mean_12",
    "usage_rolling_mean_96",
    "usage_rolling_std_12",
    "usage_rolling_std_96",
    "time_sin",
    "time_cos",
    "hour",
    "minute",
    "day_of_week_num"
]
TARGET = "target_next_15m_kWh"

X = df[FEATURES].copy()
y = df[TARGET].astype(float).copy()

n = len(df)
train_end = int(n * 0.64)
val_end = int(n * 0.80)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]
test_dates = df["date"].iloc[val_end:].reset_index(drop=True)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Final test:", len(X_test))
print("Test period:", test_dates.min(), "→", test_dates.max())


In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    [
        ("num", "passthrough", numeric_cols),
        ("cat", encoder, categorical_cols)
    ],
    remainder="drop"
)

X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("Processed features:", len(feature_names))

## 5. Model Selection

In [ ]:
models = {
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=350, min_samples_leaf=2, max_features=0.90,
        random_state=42, n_jobs=-1
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2, max_features=0.90,
        random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.04, max_leaf_nodes=31,
        l2_regularization=0.5, random_state=42
    ),
    "Ridge": Ridge(alpha=3.0)
}

try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.90,
        min_child_weight=3,
        reg_lambda=2.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )
    xgb_status = "available"
except Exception:
    xgb_status = "not available"

validation_rows = []
naive_val_pred = X_val["Usage_kWh"].to_numpy(dtype=float)
validation_rows.append({
    "Model": "Naive Current Usage",
    "RMSE": rmse(y_val, naive_val_pred),
    "MAE": mean_absolute_error(y_val, naive_val_pred),
    "R2": r2_score(y_val, naive_val_pred)
})

for name, model in models.items():
    model.fit(X_train_p, y_train)
    pred = model.predict(X_val_p)
    validation_rows.append({
        "Model": name,
        "RMSE": rmse(y_val, pred),
        "MAE": mean_absolute_error(y_val, pred),
        "R2": r2_score(y_val, pred)
    })

validation_df = pd.DataFrame(validation_rows).sort_values("RMSE").reset_index(drop=True)
model_candidates = validation_df[validation_df["Model"] != "Naive Current Usage"].reset_index(drop=True)
best_model_name = model_candidates.loc[0, "Model"]

print(f"XGBoost: {xgb_status}")
display(validation_df.round(3))

fig = px.bar(
    validation_df.sort_values("RMSE", ascending=True),
    x="RMSE", y="Model", orientation="h", text="RMSE",
    color="RMSE", color_continuous_scale="Blues"
)
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
style_fig(fig, "Chronological Validation — Lower RMSE Is Better", 470)
fig.update_xaxes(title="Validation RMSE (kWh)")
fig.update_yaxes(title="")
fig.show()


## 6. Final Holdout Evaluation

In [ ]:
selected_model = models[best_model_name]

X_pretest = np.vstack([X_train_p, X_val_p])
y_pretest = pd.concat([y_train, y_val], ignore_index=True)

selected_model.fit(X_pretest, y_pretest)
y_pred_test = selected_model.predict(X_test_p)

test_rmse = rmse(y_test, y_pred_test)
test_mae = float(mean_absolute_error(y_test, y_pred_test))
test_r2 = float(r2_score(y_test, y_pred_test))
prediction_bias = float(np.mean(y_pred_test - y_test.to_numpy()))
mean_actual = float(np.mean(y_test))
normalized_rmse = test_rmse / mean_actual if mean_actual else np.nan
normalized_mae = test_mae / mean_actual if mean_actual else np.nan

print(f"Selected model: {best_model_name}")
print(f"Test RMSE: {test_rmse:.2f} kWh")
print(f"Test MAE: {test_mae:.2f} kWh")
print(f"Test R²: {test_r2:.4f}")
print(f"Normalized RMSE: {normalized_rmse:.3f}")
print(f"Normalized MAE: {normalized_mae:.3f}")
print(f"Prediction bias: {prediction_bias:.3f} kWh")


In [ ]:
actual = y_test.to_numpy(dtype=float)
predicted = np.asarray(y_pred_test, dtype=float)
residuals = actual - predicted

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Actual vs Predicted", "Residuals vs Predicted")
)

fig.add_trace(
    go.Scatter(
        x=actual, y=predicted, mode="markers",
        marker=dict(size=5, opacity=0.45, color=PALETTE["blue"]),
        name="Prediction"
    ),
    row=1, col=1
)

lo = min(actual.min(), predicted.min())
hi = max(actual.max(), predicted.max())

fig.add_trace(
    go.Scatter(
        x=[lo, hi], y=[lo, hi], mode="lines",
        line=dict(dash="dash", color=PALETTE["red"]),
        name="Perfect prediction"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=predicted, y=residuals, mode="markers",
        marker=dict(size=5, opacity=0.45, color=PALETTE["teal"]),
        name="Residual"
    ),
    row=1, col=2
)

fig.add_hline(y=0, line_dash="dash", line_color=PALETTE["red"], row=1, col=2)

fig.update_xaxes(title="Actual (kWh)", row=1, col=1)
fig.update_yaxes(title="Predicted (kWh)", row=1, col=1)
fig.update_xaxes(title="Predicted (kWh)", row=1, col=2)
fig.update_yaxes(title="Actual − Predicted (kWh)", row=1, col=2)

style_fig(fig, f"Final Holdout Diagnostics — {best_model_name}", 500)
fig.show()

### 6.1 Why These Features Matter

Feature importance answers **which available signals the model relied on most**. It is not causality. The table below translates the top model drivers into operational meaning rather than listing importance values alone.


In [ ]:
importance = None

if hasattr(selected_model, "feature_importances_"):
    importance = pd.Series(selected_model.feature_importances_, index=feature_names).sort_values(ascending=False)
elif hasattr(selected_model, "coef_"):
    importance = pd.Series(np.abs(selected_model.coef_), index=feature_names).sort_values(ascending=False)

if importance is not None:
    top_importance = importance.head(15).sort_values()
    fig = px.bar(
        top_importance,
        x=top_importance.values,
        y=top_importance.index,
        orientation="h",
        text=top_importance.values,
        color=top_importance.values,
        color_continuous_scale="Blues"
    )
    fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
    style_fig(fig, f"Feature Importance — Top Forecast Drivers ({best_model_name})", 570)
    fig.update_xaxes(title="Relative Importance")
    fig.update_yaxes(title="")
    fig.show()

    importance_table = importance.head(15).rename("Importance").to_frame().reset_index()
    importance_table.columns = ["Feature", "Importance"]
    display(importance_table.round(4))

    reason_map = {
        "Usage_kWh": "Current load is the strongest short-horizon reference.",
        "usage_current": "Latest observed demand level directly anchors the next interval.",
        "usage_lag_1": "Captures immediate demand persistence.",
        "usage_lag_4": "Captures the recent one-hour operating cycle.",
        "usage_lag_12": "Captures the recent three-hour operating pattern.",
        "usage_lag_96": "Captures the same time-of-day behaviour from the prior day.",
        "CO2(tCO2)": "Tracks operational intensity associated with energy consumption.",
        "Lagging_Current_Reactive.Power_kVarh": "Reflects electrical loading behaviour.",
        "Lagging_Current_Power_Factor": "Shows how efficiently electrical load is being used.",
        "usage_rolling_mean_4": "Smooths short-term noise and captures the current load regime.",
        "usage_rolling_mean_12": "Captures the recent operating level.",
        "usage_rolling_mean_96": "Captures the daily baseline.",
        "usage_rolling_std_12": "Measures short-term volatility.",
        "usage_rolling_std_96": "Measures daily variability.",
        "time_sin": "Encodes time-of-day cyclically.",
        "time_cos": "Complements cyclic time encoding.",
        "hour": "Captures hour-specific production behaviour.",
        "minute": "Captures the 15-minute position within the hour.",
        "NSM": "Represents seconds from midnight and operating time position.",
        "Load_Type": "Captures the plant's operating demand category."
    }
    explanation = importance_table.copy()
    explanation["Why it matters"] = explanation["Feature"].map(
        lambda x: reason_map.get(x.replace("num__", "").replace("cat__", "").replace("_", "_"), "Model-selected signal capturing operating state or timing.")
    )
    display(explanation)


## 7. Operational Early-Warning Layer

In [ ]:
HIGH_DEMAND_THRESHOLD = 80.918

forecast = pd.DataFrame({
    "Timestamp": test_dates,
    "Actual": actual,
    "Predicted": predicted
})

forecast["Threshold"] = HIGH_DEMAND_THRESHOLD
forecast["Actual_High"] = forecast["Actual"] >= HIGH_DEMAND_THRESHOLD
forecast["Predicted_High"] = forecast["Predicted"] >= HIGH_DEMAND_THRESHOLD
forecast["Alert"] = np.where(forecast["Predicted_High"], "High-Demand Risk", "Normal")

tp = int(((forecast["Actual_High"]) & (forecast["Predicted_High"])).sum())
fn = int(((forecast["Actual_High"]) & (~forecast["Predicted_High"])).sum())
fp = int(((~forecast["Actual_High"]) & (forecast["Predicted_High"])).sum())
tn = int(((~forecast["Actual_High"]) & (~forecast["Predicted_High"])).sum())

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

alert_kpis = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "Correct High-Demand", "Missed High-Demand", "False Alert"],
    "Value": [precision, recall, f1, tp, fn, fp]
})
display(alert_kpis)

In [ ]:
# Executive early-warning view on the final holdout
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=forecast["Timestamp"], y=forecast["Actual"],
    mode="lines", name="Actual",
    line=dict(color=PALETTE["navy"], width=1.6)
))

fig.add_trace(go.Scatter(
    x=forecast["Timestamp"], y=forecast["Predicted"],
    mode="lines", name="Forecast",
    line=dict(color=PALETTE["blue"], width=1.8)
))

fig.add_trace(go.Scatter(
    x=forecast["Timestamp"], y=forecast["Threshold"],
    mode="lines", name="High-Demand Threshold",
    line=dict(color=PALETTE["red"], dash="dash", width=2)
))

alerts = forecast[forecast["Predicted_High"]]
fig.add_trace(go.Scatter(
    x=alerts["Timestamp"], y=alerts["Predicted"],
    mode="markers", name="Early Warning",
    marker=dict(size=7, color=PALETTE["orange"], symbol="diamond")
))

style_fig(fig, "Industrial Energy Early-Warning Dashboard — Final Holdout", 560)
fig.update_yaxes(title="Next-15-Minute Energy (kWh)")
fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [ ]:
# High-demand confusion matrix
cm = np.array([[tn, fp], [fn, tp]])

fig = px.imshow(
    cm,
    text_auto=True,
    x=["Predicted Normal", "Predicted High-Demand"],
    y=["Actual Normal", "Actual High-Demand"],
    color_continuous_scale="Blues",
    aspect="auto"
)
style_fig(fig, "High-Demand Warning Confusion Matrix", 430)
fig.update_layout(coloraxis_colorbar_title="Intervals")
fig.show()

In [ ]:
# Forecast performance by operating load type
test_eval = df.iloc[val_end:].copy()
test_eval = test_eval.reset_index(drop=True)
test_eval["Actual_Next"] = actual
test_eval["Predicted_Next"] = predicted
test_eval["Abs_Error"] = np.abs(test_eval["Actual_Next"] - test_eval["Predicted_Next"])

load_error = (
    test_eval.groupby("Load_Type", as_index=False)
    .agg(
        MAE=("Abs_Error","mean"),
        Actual_Mean=("Actual_Next","mean"),
        Predicted_Mean=("Predicted_Next","mean"),
        Intervals=("Actual_Next","size")
    )
)

display(load_error.round(2))

fig = px.bar(
    load_error,
    x="Load_Type",
    y="MAE",
    text="MAE",
    color="MAE",
    color_continuous_scale="Blues"
)
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
style_fig(fig, "Forecast Error by Operating Load Type", 440)
fig.update_yaxes(title="MAE (kWh)")
fig.show()

## 8. Forward 24-Step Operational Scenario

This is a **scenario forecast**, not a guaranteed future measurement. Future sensor inputs are held at the latest observed operating state while the model recursively updates predicted energy.


In [ ]:
# Recursive 24-step scenario using latest observed exogenous measurements.
future_steps = 24
last_row = df.iloc[-1].copy()
history_usage = list(df["Usage_kWh"].tail(12).astype(float))

future_rows = []

for step in range(1, future_steps + 1):
    ts = last_row["date"] + pd.Timedelta(minutes=15 * step)

    hour = ts.hour
    minute = ts.minute
    dow = ts.dayofweek
    minutes_midnight = hour * 60 + minute

    row = last_row.copy()
    row["date"] = ts
    row["hour"] = hour
    row["minute"] = minute
    row["day_of_week_num"] = dow
    row["WeekStatus"] = "Weekend" if dow >= 5 else "Weekday"
    row["Day_of_week"] = ts.day_name()

    row["time_sin"] = np.sin(2 * np.pi * minutes_midnight / 1440)
    row["time_cos"] = np.cos(2 * np.pi * minutes_midnight / 1440)

    row["usage_lag_1"] = history_usage[-1]
    row["usage_lag_4"] = history_usage[-4]
    row["usage_lag_12"] = history_usage[-12]
    row["usage_rolling_mean_4"] = np.mean(history_usage[-4:])
    row["usage_rolling_mean_12"] = np.mean(history_usage[-12:])
    row["usage_rolling_std_12"] = np.std(history_usage[-12:], ddof=1)

    future_x = pd.DataFrame([row[FEATURES]])
    future_x_p = preprocessor.transform(future_x)
    forecast_value = float(selected_model.predict(future_x_p)[0])

    future_rows.append([ts, forecast_value])
    history_usage.append(forecast_value)

future_24 = pd.DataFrame(future_rows, columns=["Timestamp", "Forecast_kWh"])
future_24["Threshold"] = HIGH_DEMAND_THRESHOLD
future_24["Risk"] = np.where(
    future_24["Forecast_kWh"] >= HIGH_DEMAND_THRESHOLD,
    "High-Demand Risk",
    "Normal"
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=future_24["Timestamp"],
    y=future_24["Forecast_kWh"],
    mode="lines+markers",
    name="Scenario Forecast",
    line=dict(color=PALETTE["blue"], width=3),
    marker=dict(size=6)
))

fig.add_trace(go.Scatter(
    x=future_24["Timestamp"],
    y=future_24["Threshold"],
    mode="lines",
    name="High-Demand Threshold",
    line=dict(color=PALETTE["red"], dash="dash", width=2)
))

risk_points = future_24[future_24["Risk"] == "High-Demand Risk"]
fig.add_trace(go.Scatter(
    x=risk_points["Timestamp"],
    y=risk_points["Forecast_kWh"],
    mode="markers",
    name="High-Demand Risk",
    marker=dict(color=PALETTE["orange"], size=9, symbol="diamond")
))

style_fig(fig, "Next 24 Intervals — Forward Energy Demand Scenario", 520)
fig.update_yaxes(title="Forecast Energy (kWh)")
fig.show()

display(future_24.round(2))

## 9. Tableau-Ready Business Output

The final holdout predictions are exported as a flat table so Tableau can consume the result directly. One row represents one 15-minute interval.

**Business fields:** actual demand, predicted demand, absolute error, percentage error, high-demand threshold, predicted risk, actual risk, warning outcome, and load context.


In [ ]:
# Build the complete Tableau-ready holdout output.
tableau_output = forecast.copy()
tableau_output["Absolute_Error_kWh"] = np.abs(tableau_output["Actual"] - tableau_output["Predicted"])
tableau_output["Signed_Error_kWh"] = tableau_output["Predicted"] - tableau_output["Actual"]
tableau_output["Absolute_Percent_Error"] = np.where(
    tableau_output["Actual"] != 0,
    tableau_output["Absolute_Error_kWh"] / np.abs(tableau_output["Actual"]),
    np.nan
)
tableau_output["Warning_Outcome"] = np.select(
    [tableau_output["Predicted_High"] & tableau_output["Actual_High"],
     tableau_output["Predicted_High"] & ~tableau_output["Actual_High"],
     ~tableau_output["Predicted_High"] & tableau_output["Actual_High"]],
    ["Correct High-Demand Warning", "False Alert", "Missed High-Demand"],
    default="Correct Normal"
)

test_context = df.iloc[val_end:][["Load_Type", "WeekStatus", "Day_of_week", "hour", "minute"]].reset_index(drop=True)
tableau_output = pd.concat([tableau_output.reset_index(drop=True), test_context], axis=1)
tableau_output["Model"] = best_model_name
tableau_output["Forecast_Horizon"] = "Next 15 Minutes"

tableau_path = "/content/industrial_energy_forecast_tableau.csv"
tableau_output.to_csv(tableau_path, index=False)

display(tableau_output.sort_values("Predicted", ascending=False).head(20).round(3))
print(f"Tableau-ready file: {tableau_path}")
print(f"Rows exported: {len(tableau_output):,}")


## 10. Executive KPI Dashboard

The dashboard separates **forecast quality** from **operational risk detection** so a decision-maker can assess model performance and business usefulness at a glance.


In [ ]:
# 10. Executive KPI Dashboard

kpi_values = [
    ("Test RMSE", f"{test_rmse:.2f} kWh", "Forecast Error", "#2563EB", "#EFF6FF"),
    ("Test MAE", f"{test_mae:.2f} kWh", "Forecast Error", "#0891B2", "#ECFEFF"),
    ("Test R²", f"{test_r2:.3f}", "Model Fit", "#7C3AED", "#F5F3FF"),
    ("Normalized RMSE", f"{normalized_rmse:.3f}", "Relative Error", "#0F766E", "#F0FDFA"),
    ("High-Demand Recall", f"{recall:.1%}", "Risk Detection", "#DC2626", "#FEF2F2"),
    ("High-Demand Precision", f"{precision:.1%}", "Alert Quality", "#EA580C", "#FFF7ED"),
    ("High-Demand F1", f"{f1:.1%}", "Risk Performance", "#CA8A04", "#FEFCE8"),
    ("Prediction Bias", f"{prediction_bias:.2f} kWh", "Calibration", "#475569", "#F1F5F9")
]

fig = go.Figure()
positions = [
    (0.14, 0.70), (0.38, 0.70), (0.62, 0.70), (0.86, 0.70),
    (0.14, 0.28), (0.38, 0.28), (0.62, 0.28), (0.86, 0.28)
]

for (label, value, group, accent, fill), (x, y) in zip(kpi_values, positions):
    # Main KPI card
    fig.add_shape(
        type="rect", x0=x-0.105, x1=x+0.105, y0=y-0.145, y1=y+0.145,
        xref="paper", yref="paper",
        line=dict(color="#E2E8F0", width=1),
        fillcolor=fill, layer="below"
    )
    # Accent strip
    fig.add_shape(
        type="rect", x0=x-0.105, x1=x+0.105, y0=y+0.115, y1=y+0.145,
        xref="paper", yref="paper",
        line=dict(width=0), fillcolor=accent, layer="above"
    )
    fig.add_annotation(
        x=x, y=y+0.065, xref="paper", yref="paper",
        text=f"<b>{group}</b>", showarrow=False,
        font=dict(size=10, color=accent)
    )
    fig.add_annotation(
        x=x, y=y-0.005, xref="paper", yref="paper",
        text=f"<b>{value}</b>", showarrow=False,
        font=dict(size=24, color="#0F172A")
    )
    fig.add_annotation(
        x=x, y=y-0.082, xref="paper", yref="paper",
        text=label, showarrow=False,
        font=dict(size=11, color="#475569")
    )

# Section labels for quick executive reading
fig.add_annotation(
    x=0.02, y=0.89, xref="paper", yref="paper",
    text="<b>FORECAST QUALITY</b>", showarrow=False, xanchor="left",
    font=dict(size=12, color="#334155")
)
fig.add_annotation(
    x=0.02, y=0.47, xref="paper", yref="paper",
    text="<b>OPERATIONAL RISK & CALIBRATION</b>", showarrow=False, xanchor="left",
    font=dict(size=12, color="#334155")
)

fig.update_xaxes(visible=False, range=[0, 1])
fig.update_yaxes(visible=False, range=[0, 1])
fig.update_layout(
    template=PLOT_TEMPLATE,
    height=560,
    title=dict(
        text="Industrial Energy Forecast — Executive KPI Dashboard",
        x=0.02, xanchor="left",
        font=dict(size=22, color="#0F172A")
    ),
    margin=dict(l=20, r=20, t=85, b=20),
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",
    showlegend=False
)
fig.show()


In [ ]:
final_kpis = pd.DataFrame({
    "KPI": [
        "Selected Model",
        "Validation RMSE (kWh)",
        "Test RMSE (kWh)",
        "Test MAE (kWh)",
        "Test R²",
        "Prediction Bias (kWh)",
        "High-Demand Threshold (kWh)",
        "High-Demand Precision",
        "High-Demand Recall",
        "High-Demand F1"
    ],
    "Value": [
        best_model_name,
        validation_df.loc[0, "RMSE"],
        test_rmse,
        test_mae,
        test_r2,
        prediction_bias,
        HIGH_DEMAND_THRESHOLD,
        precision,
        recall,
        f1
    ]
})
display(final_kpis)

# 11. Final Business Solution

**Forecast → Detect Risk → Prioritize → Tableau → Operational Review → Measure**

- Forecast the next 15-minute energy requirement for every holdout interval.
- Use the model only after validation beats the naive current-usage baseline.
- Flag predicted demand at or above **80.918 kWh** for operational review.
- Use the Tableau-ready output to rank high-demand periods, inspect error, and review load context.
- Reconcile actual vs predicted demand; persistent error drift becomes a retraining signal.
- Automated equipment control is outside this model and requires plant engineering and safety approval.


# Final Deliverable

The notebook produces three decision layers: **next-15-minute prediction**, **high-demand early warning**, and **Tableau-ready operational output**. Regression quality is reported with RMSE, MAE, R², and normalized error ratios; the model is selected from a chronological validation set rather than by forcing a low error score.


In [ ]:
# Final notebook integrity check
required_objects = [
    "df", "X_train", "X_val", "X_test", "preprocessor", "validation_df",
    "selected_model", "y_pred_test", "forecast", "tableau_output", "future_24",
    "HIGH_DEMAND_THRESHOLD", "precision", "recall", "f1", "normalized_rmse"
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError("Integrity check failed: " + ", ".join(missing_objects))

print("Notebook integrity check: PASSED")
print(f"Selected model: {best_model_name}")
print(f"Final test RMSE: {test_rmse:.2f} kWh")
print(f"Final test R²: {test_r2:.3f}")
print(f"Normalized RMSE: {normalized_rmse:.3f}")
print(f"High-demand recall: {recall:.1%}")
print(f"Tableau rows exported: {len(tableau_output):,}")
